# 01 — Exploratory Data Analysis
## Smart Irrigation for Tomato (FYP)

This notebook inspects the Kathmandu IoT sensor workbook used by the ESP32 node (DHT11 air temperature/humidity, analog soil moisture, Kathmandu air pressure, pump ON/OFF).

**Questions**
1. Are the readings physically plausible for Kathmandu tomato weather?
2. What actually drives the historical pump?
3. How should a *tomato-specific* irrigation label differ from that pump?
4. Which engineered FAO-56 features (VPD, ET0 proxy) are worth giving the model?



In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.preprocess import PROCESSED_CSV, main as preprocess_main
from src.generate_season import OUT_CSV as SEASON_CSV, main as generate_season_main
from src.eda import run_eda

sns.set_theme(style="whitegrid", context="notebook")
if not PROCESSED_CSV.exists():
    preprocess_main()
if not SEASON_CSV.exists():
    generate_season_main()

df = pd.read_csv(PROCESSED_CSV)
season = pd.read_csv(SEASON_CSV, parse_dates=["timestamp"])
df.head()



## Data quality

3,000 complete rows. No missing values. Soil ADC is mapped to 0–100% so it matches the ESP32 firmware (`soilMoisture` in the Google Sheets payload). Pressure (845–865 hPa) is consistent with Kathmandu at ~1,400 m, not sea-level 1013 hPa.



In [ ]:
print("rows, cols:", df.shape)
print(df.dtypes)
print("\nmissing:\n", df.isna().sum())
df[["temperature", "humidity", "pressure", "soilMoisture", "vpd_kpa", "pump_historical", "irrigate"]].describe().round(2)



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
specs = [
    ("soilMoisture", "Soil moisture (%)"),
    ("temperature", "Air temperature (C)"),
    ("humidity", "Relative humidity (%)"),
    ("pressure", "Pressure (hPa)"),
]
for ax, (col, label) in zip(axes.ravel(), specs):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="#2A6F97")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
fig.suptitle("Kathmandu IoT sensor distributions (n=3000)")
fig.tight_layout()



## What the historical pump is doing

Pump ON rate is 52.3%. Correlation of soil moisture with the pump is about **-0.85**. Temperature, humidity, and pressure are almost uncorrelated with the pump. The old controller is a **soil-moisture threshold**, which matches the ESP32 firmware (`relay ON` when soil is at the dry stop).



In [ ]:
print("pump ON rate:", df["pump_historical"].mean().round(3))
print("tomato irrigate rate:", df["irrigate"].mean().round(3))
print("agreement pump vs tomato label:", (df["pump_historical"] == df["irrigate"]).mean().round(3))
print()
print(df[["soilMoisture", "temperature", "humidity", "pressure", "vpd_kpa", "pump_historical", "irrigate"]].corr().round(3))



In [ ]:
sample = df.sample(800, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
sns.scatterplot(data=sample, x="soilMoisture", y="temperature", hue="pump_historical",
                palette=["#ADB5BD", "#C1121F"], s=22, ax=axes[0], edgecolor=None)
axes[0].set_title("Historical pump")
axes[0].set_xlabel("Soil moisture (%)")
axes[0].set_ylabel("Air temperature (C)")
sns.scatterplot(data=sample, x="soilMoisture", y="temperature", hue="irrigate",
                palette=["#ADB5BD", "#01497C"], s=22, ax=axes[1], edgecolor=None)
axes[1].set_title("FAO tomato irrigation label")
axes[1].set_xlabel("Soil moisture (%)")
fig.suptitle("Same sensors, two policies")
fig.tight_layout()



## Tomato-specific label (FAO-56)

Tomato management allowed depletion is about **0.40**, so irrigation should start near 55–60% relative moisture — *sooner* on high vapor-pressure deficit (hot, dry air) and *not at all* when the soil is already wet.

`vpd_kpa` is computed with the Tetens/FAO-56 saturation vapor pressure formula from DHT11 temperature and humidity. That is the climate signal the historical pump ignored.

The tomato label agrees with the historical pump on **92.2%** of rows. The disagreements are the FYP contribution: irrigate earlier in heat/dry air, hold off when evaporative demand is low.



In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
sns.scatterplot(data=df.sample(900, random_state=42), x="soilMoisture", y="vpd_kpa",
                hue="irrigate", palette=["#ADB5BD", "#01497C"], s=24, ax=ax, edgecolor=None)
ax.set_xlabel("Soil moisture (%)")
ax.set_ylabel("Vapor pressure deficit (kPa)")
ax.set_title("Irrigate when soil is dry and VPD is high")



## Simulated Kathmandu tomato season

The workbook has **no timestamps**. To inspect diurnal cycle and FAO crop stages we simulate a 135-day spring crop at the firmware's **15-minute** upload interval, using FAO-56 tomato crop coefficients (Kc 0.60 → 1.15 → 0.80) and a Kathmandu temperature/humidity climatology. This file is for EDA only; **the production model is trained on the 3,000 real logs.**



In [ ]:
week = season.iloc[:96 * 7]
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(week["timestamp"], week["temperature"], color="#C1121F", lw=1)
axes[0].set_ylabel("Temp (C)")
axes[1].plot(week["timestamp"], week["soilMoisture"], color="#2A6F97", lw=1)
axes[1].set_ylabel("Soil moisture (%)")
axes[2].fill_between(week["timestamp"], 0, week["irrigate"], color="#01497C", step="mid", alpha=0.75)
axes[2].set_ylabel("Irrigate")
axes[2].set_xlabel("Timestamp")
fig.suptitle("Simulated Kathmandu tomato week (15-min interval)")
fig.tight_layout()

print(season.groupby("growth_stage", observed=False)["irrigate"].mean())



## EDA conclusions for modeling

| Finding | Implication |
| --- | --- |
| No missing values, Kathmandu pressure is realistic | Little cleaning needed |
| Historical pump ≈ soil threshold | A 55% moisture cutoff is the baseline, not the goal |
| Temp/humidity unused by the pump | Encode them as VPD / ET0 / heat stress |
| Tomato FAO label uses soil × climate | Train the API model on `irrigate`, not `pump_historical` |
| No timestamps in the workbook | Use a stratified 80/20 split (not a time split) on the 3,000 logs |

Re-run the saved figure pack with `python -m src.eda` (writes `results/figures/`).



In [ ]:
# Uncomment to regenerate every EDA figure under results/figures/
# summary = run_eda()
# summary.keys()

